## Übung: Streaming

In diesem Assignment wird eine Stream Umgebung simuliert. Es werden synthetisch Twitter Daten generiert und diese in das Verzeichnis `data/` geladen.

Die Twitterdaten sollen anschließend mit Spark Structured Streaming eingelesen und analysiert werden.

Die Datengenerierung erfolgt über ein Python Skript, dass in einem zweiten Container läuft. 

### Vorbereitung der Daten
Überprüfen Sie, ob beim Start von docker compose das Verzeichnis `data` mit Daten befüllt wird. 

### Benötigte Imports laden
Fügen Sie hier alle benötigten Imports ein.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Daten Stream erstellen
- Erstellen Sie eine `SparkSession`.
- Legen Sie das folgende Schema an:
  - id, integer
  - text, string
  - timestamp, timestamp
- Erstellen Sie einen Streaming `DataFrame`. Sie können dazu das Beispiel in der [Dokumentation](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#creating-streaming-dataframes-and-streaming-datasets) anpassen.

- Das zu überwachende Directory ist `data`. Das einzulesende Dateiformat ist `csv`. Die Option `header` sollte auf `True` gesetzt werden.

In [2]:
spark = SparkSession.builder.appName("Praktikum2_HashtagCount").config("spark.sql.session.timeZone", "Europe/Berlin").getOrCreate()

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

tweet_df = spark.readStream.format("csv").option("path", "../data/").option("header", "true").schema(schema).load()
tweet_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



### Bearbeitung des Datastroms
Auf dem erstellten streaming DataFrame können verschiedene Operationen durchgeführt werden. Die meisten kennen Sie schon aus der DataFrame API. Für die Analyse der Streamingdaten sollen Sie die folgenden Operationen auf den eingelesenen Dateien ausführen:

- Zerlegen Sie den Text in einzelne Tokens
- Filtern Sie Hashtags
- Zählen Sie für jedes HashTag die Häufigkeit

Die Auswertung soll alle 30 Sekunden die Zusammenfassung der letzten 2 Minuten ausgeben. Dabei soll nach der Event-Zeit (timestamp) ausgewertet werden. Es sollen auch Events berücksichtigt werden, die bis zu einer Minute zu spät eingelaufen sind. Ergänzen Sie das folgende Statement:

Tipp: In der Dokumentation ist das Windowing gut beschrieben.
[Dokumentation](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time)

In [3]:
# Ersetze Sonderzeichen wie "." am Ende des Satzes
tweet_df = tweet_df.withColumn(
    'cleaned_text', 
    regexp_replace(col('text'), r"[^a-zA-Z0-9\s#]", "")
)

# Korrigiere falsche Timestamps des Tweet-Generator
tweet_df = tweet_df.withColumn("timestamp", col("timestamp") + expr("INTERVAL 2 HOURS")) 
tweet_df = tweet_df.withWatermark("timestamp", "1 minute")


hashtags = tweet_df.withColumn('hashtag', explode(split(col('cleaned_text'), ' '))) \
  .filter(col('hashtag').contains('#')) \
  .withColumn('hashtag', lower(trim(col('hashtag')))) \
  .groupBy(window(col("timestamp"), "30 seconds", "10 seconds"), col("hashtag")) \
  .count() \
  .filter(col("count") >= 20)


windowSpec = Window.partitionBy("window").orderBy(col("count").desc())

### Tweets Analyse ausgeben (1)

Schreiben Sie die Tweets in den Hauptspeicher. Mehr Informationen finden Sie [hier](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#starting-streaming-queries).

In [4]:
streamwriter = hashtags.writeStream.outputMode("update").format("memory").trigger(processingTime='10 seconds').queryName("tweets2ram").start()
print("Stream is running")

Stream is running


Tweets in Dateien schreiben:

In [5]:
output_df = hashtags.select(
    col("window.start").alias("start"),
    col("window.end").alias("end"),
    "hashtag",
    "count"
)

file_stream = output_df.writeStream \
    .outputMode("append") \
    .format("csv") \
    .option("path", "../output/relevant_hashtags/") \
    .option("checkpointLocation", "../checkpoints/hashtag_storage/") \
    .option("header", "true") \
    .trigger(processingTime='10 seconds') \
    .start()

print("Stream is running")

Stream is running


Geben Sie alle 10 Sekunden die Liste mit den Hashtags aus dem Hauptspeicher aus. Formulieren Sie dazu im Parameter von `spark.sql` eine SQL Abfrage.

In [ ]:
from IPython.display import display, clear_output
import time

# 2 Minuten lang analysieren
for i in range(12):
    clear_output(wait=True)
    display(spark.sql('SELECT * FROM tweets2ram order by window desc, count desc limit 10').show(truncate=False))
    time.sleep(10)


+------------------------------------------+-----------+-----+
|window                                    |hashtag    |count|
+------------------------------------------+-----------+-----+
|{2026-05-14 09:38:00, 2026-05-14 09:38:30}|#beans     |30   |
|{2026-05-14 09:38:00, 2026-05-14 09:38:30}|#ice       |26   |
|{2026-05-14 09:38:00, 2026-05-14 09:38:30}|#danish    |26   |
|{2026-05-14 09:38:00, 2026-05-14 09:38:30}|#lollipop  |23   |
|{2026-05-14 09:38:00, 2026-05-14 09:38:30}|#sesame    |22   |
|{2026-05-14 09:38:00, 2026-05-14 09:38:30}|#cheesecake|20   |
|{2026-05-14 09:37:50, 2026-05-14 09:38:20}|#lollipop  |34   |
|{2026-05-14 09:37:50, 2026-05-14 09:38:20}|#danish    |31   |
|{2026-05-14 09:37:50, 2026-05-14 09:38:20}|#beans     |31   |
|{2026-05-14 09:37:50, 2026-05-14 09:38:20}|#lollipop  |25   |
+------------------------------------------+-----------+-----+



None

#### Begründung Output Mode: 

Beim Schreiben der Ergebnisse in Dateien wird der Complete Mode z.B. gar nicht erst unterstützt, da dieser zu ineffizient wäre.
Update Mode würde bei verspäteten Dateien ein neues Duplikat schreiben, Append exportiert das Ergebnis nur einmal nach dem Watermark und passt damit am besten. 

Beim Halten der Daten in einer Memory-Tabelle ist der Update Mode am besten, da dieser direkt Zwischenstände anzeigt. Außerdem wird der gesamte Speicher nicht durch das Schreiben
des gesamten Status wie bei Complete belastet.

### Stoppen des Streams

In [ ]:
streamwriter.stop()
file_stream.stop()

In [7]:
!rm -rf ../checkpoints

In [8]:
!rm -r ../output/

In [9]:
!rm -r ../data/

rm: remove write-protected regular file '../data/twitter_0.csv'? ^C


In [ ]:
p.kill()